# Phase 2 — threshold calibration and held-out evaluation
## Goal
Select a threshold from development captures only, then freeze it before subject-disjoint final evaluation. **Three consented single images are available, but they provide no valid enrollment/probe pairs. The published outcome is INSUFFICIENT_DATA and the operational threshold remains null.**

FMR = accepted impostor comparisons / all impostor comparisons. FNMR = rejected genuine comparisons / all genuine comparisons. These are verification metrics, not expression-classifier Macro F1.

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Protocol and prespecified tradeoff
Adult volunteers use pseudonyms with 3–5 enrollment images and independent-session probes. Development and test subjects, image paths and decoded content hashes must not overlap. No RAF-DB identity assumptions. Repeated pair matrices are dependent: development selection uses a deterministic disjoint-subject impostor subset selected without inspecting its scores.

Default target FMR is 1% with a one-sided 95% order-statistic tolerance bound. With zero errors this needs **299 independent impostor comparisons**, each using two different subjects without re-use. A handful of volunteers cannot satisfy this policy. This conservative research gate intentionally keeps acceptance disabled for small cohorts. Sample size does not establish representativeness, fairness or production readiness.

In [ ]:
from face_verification.calibration import calibrate_development
report = calibrate_development([], 'NO_REAL_MODEL_CALIBRATION', config)
print(json.dumps(report,indent=2))
assert report['status'] == 'INSUFFICIENT_DATA' and report['threshold'] is None

## Synthetic score visualization — algorithm demonstration only
The following deterministic toy scores demonstrate plotting and confusion denominators. They do not come from faces and cannot authorize any verification threshold. Curves describe scores; they are not evidence of real biometric accuracy.

In [ ]:
from face_verification.calibration import Pair,confusion_metrics
from face_verification.plotting import plot_scores
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
toy_pairs = []
for i in range(60):
    a,b = f'toy_{i}_a',f'toy_{i}_b'
    hashes = tuple(f'{a}_enrollment_{j}' for j in range(3))
    toy_pairs.append(Pair(float(np.clip(rng.normal(.68,.14),-1,1)),True,a,a,hashes,f'{a}_probe','enrollment','probe'))
    toy_pairs.append(Pair(float(np.clip(rng.normal(.30,.18),-1,1)),False,a,b,hashes,f'{b}_probe','enrollment','probe'))
toy_calibration = calibrate_development(toy_pairs,'synthetic',config,data_kind='synthetic')
print('Synthetic calibration status:',toy_calibration['status'],'threshold:',toy_calibration['threshold'])
fig = plot_scores(toy_pairs,REPORTS/'synthetic_score_demo.png',title='SYNTHETIC TEST DATA — no face-model accuracy claim')
plt.show()

## Real protocol evaluation — manual opt-in
The reusable evaluator validates metadata, generates normalized embeddings without saving them, calibrates development, and evaluates the frozen cutoff on held-out subjects. Independent-subject exact intervals are reported alongside all-pair descriptive metrics. If data are inadequate, it writes a null threshold to disable verification. Test scores never influence selection.

In [ ]:
if RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT and PRIVATE_MANIFEST.exists():
    command = [sys.executable,str(ROOT/'scripts/phase2_evaluate.py'),'--manifest',str(PRIVATE_MANIFEST),'--confirm-consented-adult-data']
    completed = subprocess.run(command,cwd=ROOT,capture_output=True,text=True,timeout=3600)
    print(completed.stdout)
    assert completed.returncode == 0, 'Private evaluation failed; inspect the local error without publishing participant paths.'
else:
    print('INSUFFICIENT_DATA: real development/test comparison counts = 0; FMR/FNMR/threshold are null.')

## Checks and next steps
Exact duplicate captures are rejected; session independence relies on truthful capture metadata, and near-duplicate videos need manual review. Broad demographic, lighting, camera and pose coverage has not been measured. Do not infer subgroup guarantees, anti-spoof performance, or production reliability from toy plots or small private cohorts.

## Latest calibration eligibility

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
batch = json.loads((REPORTS/'batch02_results.json').read_text())
print(json.dumps({'status':batch['calibration_status'],'reason':batch['reason'],
    'threshold':batch['threshold'],'FMR':batch['fmr'],'FNMR':batch['fnmr'],
    'same_session_descriptive_scores':batch['descriptive_scores']},indent=2))
assert batch['calibration_status']=='INSUFFICIENT_DATA' and batch['threshold'] is None

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
print(json.dumps({'status':latest['calibration_status'],'reason':latest['reason'],'genuine_diagnostic_comparisons':latest['genuine_diagnostic_comparisons'],'impostor_diagnostic_comparisons':latest['impostor_diagnostic_comparisons'],'threshold':latest['threshold'],'FMR':latest['fmr'],'FNMR':latest['fnmr']},indent=2))
assert latest['threshold'] is None